In [1]:
import os
import re
import time
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FROM_TRAIN = 0.25
DATA_PATH = "dataset/wine_quality_merged.csv"
RESULTS_DIR = "ANN_classification_results"

HIDDEN_LAYER_COUNTS = list(range(1, 9))
NEURONS_PER_LAYER = 64
ACTIVATIONS = ["relu", "tanh", "logistic"]
LEARNING_RATES = [0.0005, 0.005, 0.05]

MAX_ITER = 300
TOL = 1e-4

os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 75)
print("ANN CLASSIFICATION - WINE QUALITY")
print("72 configurations | 18 final plot images")
print("=" * 75)


ANN CLASSIFICATION - WINE QUALITY
72 configurations | 18 final plot images


In [3]:
if not os.path.exists(DATA_PATH):
    if os.path.exists("wine_quality_merged.csv"):
        DATA_PATH = "wine_quality_merged.csv"
    else:
        raise FileNotFoundError(
            "Dataset not found. Place wine_quality_merged.csv in a "
            "'dataset' folder or beside this script."
        )

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

if df.isnull().any().any():
    raise ValueError("Missing values detected. Handle them before training.")

if "quality" not in df.columns:
    raise ValueError("Target column 'quality' was not found.")

print("\nShape after duplicate removal:", df.shape)
print("\nClass distribution:\n", df["quality"].value_counts().sort_index())


Original dataset shape: (6497, 13)

Missing values:
 fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
type                    0
dtype: int64

Duplicate rows: 1177

Shape after duplicate removal: (5320, 13)

Class distribution:
 quality
3      30
4     206
5    1752
6    2323
7     856
8     148
9       5
Name: count, dtype: int64


In [4]:
X = df.drop(columns=["quality"])
y = df["quality"]
class_labels = sorted(y.unique())

categorical_features = ["type"] if "type" in X.columns else []
numerical_features = [c for c in X.columns if c not in categorical_features]

print("\nNumerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Target classes:", class_labels)


Numerical features: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Categorical features: ['type']
Target classes: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [5]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=VALIDATION_SIZE_FROM_TRAIN,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))


Training samples: 3192
Validation samples: 1064
Testing samples: 1064


In [6]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", encoder, categorical_features)
    ]
)


In [7]:
print("\nPreparing t-SNE visualization...")
X_visual = clone(preprocessor).fit_transform(X)

try:
    tsne = TSNE(
        n_components=2, perplexity=30, max_iter=1000,
        random_state=RANDOM_STATE, init="pca", learning_rate="auto"
    )
except TypeError:
    tsne = TSNE(
        n_components=2, perplexity=30, n_iter=1000,
        random_state=RANDOM_STATE, init="pca", learning_rate="auto"
    )

X_tsne = tsne.fit_transform(X_visual)

plt.figure(figsize=(10, 7))
class_codes = pd.Categorical(y, categories=class_labels).codes
scatter = plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=class_codes, cmap="tab10", s=18, alpha=0.7
)
handles, _ = scatter.legend_elements()
plt.legend(handles, [str(v) for v in class_labels],
           title="Wine Quality", loc="best")
plt.title("2D t-SNE Visualization of Wine Quality Dataset")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "01_tsne_visualization.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("t-SNE completed.")


Preparing t-SNE visualization...
t-SNE completed.


In [8]:
def safe_filename(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))

In [9]:
experiment_results = []
validation_predictions = {}
validation_loss_curves = {}

total_start = time.perf_counter()

for hidden_count in HIDDEN_LAYER_COUNTS:
    hidden_layers = tuple([NEURONS_PER_LAYER] * hidden_count)

    for activation in ACTIVATIONS:
        for learning_rate in LEARNING_RATES:

            model_name = (
                f"ANN_L{hidden_count}_{activation}_LR{learning_rate}"
            )
            print("\n" + "-" * 72)
            print("Running:", model_name)

            model = MLPClassifier(
                hidden_layer_sizes=hidden_layers,
                activation=activation,
                solver="adam",
                learning_rate="constant",
                learning_rate_init=learning_rate,
                max_iter=MAX_ITER,
                tol=TOL,
                early_stopping=False,
                random_state=RANDOM_STATE
            )

            pipeline = Pipeline(
                steps=[
                    ("preprocessing", clone(preprocessor)),
                    ("classifier", model)
                ]
            )

            process = psutil.Process(os.getpid())
            memory_before = process.memory_info().rss / (1024 ** 2)
            cpu_before = time.process_time()
            train_start = time.perf_counter()

            pipeline.fit(X_train, y_train)

            training_time = time.perf_counter() - train_start
            cpu_time = time.process_time() - cpu_before
            memory_after = process.memory_info().rss / (1024 ** 2)
            memory_change = memory_after - memory_before
            cpu_percent = process.cpu_percent(interval=0.1)

            prediction_start = time.perf_counter()
            y_val_pred = pipeline.predict(X_val)
            prediction_time = time.perf_counter() - prediction_start

            accuracy = accuracy_score(y_val, y_val_pred)
            precision = precision_score(
                y_val, y_val_pred, average="weighted", zero_division=0
            )
            recall = recall_score(
                y_val, y_val_pred, average="weighted", zero_division=0
            )
            f1 = f1_score(
                y_val, y_val_pred, average="weighted", zero_division=0
            )

            experiment_results.append({
                "Model": model_name,
                "Hidden Layers": hidden_count,
                "Neurons per Layer": NEURONS_PER_LAYER,
                "Activation": activation,
                "Learning Rate": learning_rate,
                "Validation Accuracy": accuracy,
                "Validation Precision": precision,
                "Validation Recall": recall,
                "Validation F1-Score": f1,
                "Training Time (s)": training_time,
                "Prediction Time (s)": prediction_time,
                "CPU Time (s)": cpu_time,
                "CPU Utilization Snapshot (%)": cpu_percent,
                "Memory Before (MB)": memory_before,
                "Memory After (MB)": memory_after,
                "Memory Change (MB)": memory_change,
                "Iterations": pipeline.named_steps["classifier"].n_iter_,
                "Reached Max Iterations": (
                    pipeline.named_steps["classifier"].n_iter_ >= MAX_ITER
                )
            })

            validation_predictions[model_name] = np.asarray(y_val_pred)
            validation_loss_curves[model_name] = (
                pipeline.named_steps["classifier"].loss_curve_
            )

            print(
                f"Val Accuracy={accuracy:.4f} | "
                f"Precision={precision:.4f} | Recall={recall:.4f} | "
                f"F1={f1:.4f}"
            )
            print(
                f"Train={training_time:.2f}s | Predict={prediction_time:.3f}s | "
                f"CPU={cpu_time:.2f}s | Memory change={memory_change:.2f} MB"
            )

total_execution_time = time.perf_counter() - total_start

results_df = pd.DataFrame(experiment_results)
results_df.to_csv(
    os.path.join(RESULTS_DIR, "ann_all_72_experiments.csv"), index=False
)



------------------------------------------------------------------------
Running: ANN_L1_relu_LR0.0005
Val Accuracy=0.5470 | Precision=0.5162 | Recall=0.5470 | F1=0.5216
Train=5.30s | Predict=0.018s | CPU=7.38s | Memory change=0.70 MB

------------------------------------------------------------------------
Running: ANN_L1_relu_LR0.005
Val Accuracy=0.5320 | Precision=0.5154 | Recall=0.5320 | F1=0.5207
Train=2.16s | Predict=0.010s | CPU=3.02s | Memory change=-0.01 MB

------------------------------------------------------------------------
Running: ANN_L1_relu_LR0.05
Val Accuracy=0.5207 | Precision=0.4943 | Recall=0.5207 | F1=0.4970
Train=1.71s | Predict=0.013s | CPU=2.25s | Memory change=0.00 MB

------------------------------------------------------------------------
Running: ANN_L1_tanh_LR0.0005
Val Accuracy=0.5442 | Precision=0.5330 | Recall=0.5442 | F1=0.5157
Train=10.31s | Predict=0.014s | CPU=13.78s | Memory change=-0.01 MB

------------------------------------------------------

In [10]:
best_row = results_df.sort_values(
    by=["Validation F1-Score", "Validation Accuracy"],
    ascending=False
).iloc[0]

best_name = best_row["Model"]
best_validation_prediction = validation_predictions[best_name]
best_loss = validation_loss_curves[best_name]

print("\n" + "=" * 75)
print("BEST CONFIGURATION SELECTED USING VALIDATION WEIGHTED F1")
print("=" * 75)
print(best_row.to_string())

# Refit the selected architecture on train + validation data,
# then evaluate exactly once on the held-out test data.
best_hidden_layers = tuple(
    [NEURONS_PER_LAYER] * int(best_row["Hidden Layers"])
)

final_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=best_hidden_layers,
            activation=best_row["Activation"],
            solver="adam",
            learning_rate="constant",
            learning_rate_init=float(best_row["Learning Rate"]),
            max_iter=MAX_ITER,
            tol=TOL,
            early_stopping=False,
            random_state=RANDOM_STATE
        ))
    ]
)

final_model.fit(X_train_val, y_train_val)
test_start = time.perf_counter()
y_test_pred = final_model.predict(X_test)
best_test_prediction_time = time.perf_counter() - test_start

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(
        y_test, y_test_pred, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_test_pred, average="weighted", zero_division=0
    ),
    "F1-Score": f1_score(
        y_test, y_test_pred, average="weighted", zero_division=0
    )
}

print("\nHeld-out test-set metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")



BEST CONFIGURATION SELECTED USING VALIDATION WEIGHTED F1
Model                           ANN_L2_relu_LR0.0005
Hidden Layers                                      2
Neurons per Layer                                 64
Activation                                      relu
Learning Rate                                 0.0005
Validation Accuracy                         0.546992
Validation Precision                        0.526324
Validation Recall                           0.546992
Validation F1-Score                         0.530876
Training Time (s)                           8.381542
Prediction Time (s)                         0.017236
CPU Time (s)                               12.890625
CPU Utilization Snapshot (%)                     0.0
Memory Before (MB)                        277.898438
Memory After (MB)                         277.957031
Memory Change (MB)                          0.058594
Iterations                                       300
Reached Max Iterations                   

In [11]:
pd.DataFrame([{
    "Selected Model": best_name,
    **test_metrics,
    "Test Prediction Time (s)": best_test_prediction_time
}]).to_csv(
    os.path.join(RESULTS_DIR, "best_ann_heldout_test_metrics.csv"),
    index=False
)

pd.DataFrame({
    "Actual Quality": np.asarray(y_test),
    "Predicted Quality": np.asarray(y_test_pred)
}).to_csv(
    os.path.join(RESULTS_DIR, "best_ann_test_predictions.csv"),
    index=False
)

In [12]:
best_cm = confusion_matrix(y_test, y_test_pred, labels=class_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    best_cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title(f"Best ANN Confusion Matrix\n{best_name}")
plt.xlabel("Predicted Quality")
plt.ylabel("Actual Quality")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "03_best_ann_confusion_matrix.png"),
            dpi=300, bbox_inches="tight")
plt.close()


In [13]:
best_report_dict = classification_report(
    y_test, y_test_pred, labels=class_labels,
    zero_division=0, output_dict=True
)
best_report_text = classification_report(
    y_test, y_test_pred, labels=class_labels, zero_division=0
)

with open(
    os.path.join(RESULTS_DIR, "best_ann_classification_report.txt"),
    "w", encoding="utf-8"
) as f:
    f.write(f"Selected configuration: {best_name}\n\n")
    f.write(best_report_text)

In [14]:
best_cm = confusion_matrix(y_test, y_test_pred, labels=class_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    best_cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title(f"Best ANN Confusion Matrix\n{best_name}")
plt.xlabel("Predicted Quality")
plt.ylabel("Actual Quality")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "02_best_ann_confusion_matrix.png"),
            dpi=300, bbox_inches="tight")
plt.close()

In [15]:
best_report_dict = classification_report(
    y_test, y_test_pred, labels=class_labels,
    zero_division=0, output_dict=True
)
best_report_text = classification_report(
    y_test, y_test_pred, labels=class_labels, zero_division=0
)

with open(
    os.path.join(RESULTS_DIR, "best_ann_classification_report.txt"),
    "w", encoding="utf-8"
) as f:
    f.write(f"Selected configuration: {best_name}\n\n")
    f.write(best_report_text)

per_class_rows = []
for label in class_labels:
    row = best_report_dict[str(label)]
    per_class_rows.append({
        "Quality Class": str(label),
        "Precision": row["precision"],
        "Recall": row["recall"],
        "F1-Score": row["f1-score"],
        "Support": row["support"]
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(
    os.path.join(RESULTS_DIR, "best_ann_per_class_metrics.csv"),
    index=False
)

In [16]:
metric_map = {
    "Accuracy": "Validation Accuracy",
    "Precision": "Validation Precision",
    "Recall": "Validation Recall",
    "F1-Score": "Validation F1-Score"
}
metric_names = list(metric_map.keys())


In [17]:
for depth in HIDDEN_LAYER_COUNTS:
    subset = results_df[results_df["Hidden Layers"] == depth]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax, (metric_label, metric_column) in zip(axes, metric_map.items()):
        for activation in ACTIVATIONS:
            line_data = subset[subset["Activation"] == activation].sort_values(
                "Learning Rate"
            )
            ax.plot(
                line_data["Learning Rate"],
                line_data[metric_column],
                marker="o",
                linewidth=1.8,
                label=activation.title()
            )

        ax.set_title(metric_label)
        ax.set_xlabel("Learning Rate")
        ax.set_ylabel("Validation Score")
        ax.set_xscale("log")
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Activation", loc="upper center",
               ncol=3, bbox_to_anchor=(0.5, 1.01))
    fig.suptitle(
        f"ANN Performance with {depth} Hidden Layer(s)",
        y=1.04, fontsize=14
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(RESULTS_DIR, f"{depth+2:02d}_hidden_layers_{depth}.png"),
        dpi=300, bbox_inches="tight"
    )
    plt.close(fig)


In [18]:
for activation_index, activation in enumerate(ACTIVATIONS, start=11):
    subset = results_df[results_df["Activation"] == activation]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax, (metric_label, metric_column) in zip(axes, metric_map.items()):
        for lr in LEARNING_RATES:
            line_data = subset[subset["Learning Rate"] == lr].sort_values(
                "Hidden Layers"
            )
            ax.plot(
                line_data["Hidden Layers"],
                line_data[metric_column],
                marker="o",
                linewidth=1.8,
                label=f"LR={lr:g}"
            )

        ax.set_title(metric_label)
        ax.set_xlabel("Number of Hidden Layers")
        ax.set_ylabel("Validation Score")
        ax.set_xticks(HIDDEN_LAYER_COUNTS)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Learning Rate", loc="upper center",
               ncol=3, bbox_to_anchor=(0.5, 1.01))
    fig.suptitle(
        f"ANN Performance with {activation.title()} Activation",
        y=1.04, fontsize=14
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            RESULTS_DIR,
            f"{activation_index:02d}_activation_{activation}.png"
        ),
        dpi=300, bbox_inches="tight"
    )
    plt.close(fig)

In [19]:
for metric_index, (metric_label, metric_column) in enumerate(
    metric_map.items(), start=14
):
    top_results = results_df.nlargest(15, metric_column).sort_values(
        metric_column, ascending=True
    )

    plt.figure(figsize=(12, 7))
    plt.barh(top_results["Model"], top_results[metric_column])
    plt.title(f"Top 15 ANN Configurations - {metric_label}")
    plt.xlabel(metric_label)
    plt.ylabel("ANN Configuration")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            RESULTS_DIR, f"{metric_index:02d}_overall_{metric_label.lower()}.png"
        ),
        dpi=300, bbox_inches="tight"
    )
    plt.close()

In [20]:
resource_summary = results_df.groupby("Hidden Layers").agg({
    "Training Time (s)": "mean",
    "Prediction Time (s)": "mean",
    "CPU Time (s)": "mean",
    "Memory Change (MB)": "mean"
})

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
resource_items = [
    ("Training Time (s)", "Mean Training Time", "Seconds"),
    ("Prediction Time (s)", "Mean Prediction Time", "Seconds"),
    ("CPU Time (s)", "Mean CPU Process Time", "Seconds"),
    ("Memory Change (MB)", "Mean Memory Change", "MB")
]

for ax, (column, title, ylabel) in zip(axes.flatten(), resource_items):
    ax.plot(
        resource_summary.index, resource_summary[column],
        marker="o", linewidth=1.8
    )
    ax.set_title(title)
    ax.set_xlabel("Number of Hidden Layers")
    ax.set_ylabel(ylabel)
    ax.set_xticks(HIDDEN_LAYER_COUNTS)
    ax.grid(alpha=0.25)

fig.suptitle("ANN Computational Performance Across Hidden Layers")
fig.tight_layout()
fig.savefig(
    os.path.join(RESULTS_DIR, "18_computational_resource_comparison.png"),
    dpi=300, bbox_inches="tight"
)
plt.close(fig)

In [21]:
plot_files = [
    "01_tsne_visualization.png",
    "02_best_ann_confusion_matrix.png",
]
plot_files += [
    f"{depth+2:02d}_hidden_layers_{depth}.png"
    for depth in HIDDEN_LAYER_COUNTS
]
plot_files += [
    f"{index:02d}_activation_{activation}.png"
    for index, activation in enumerate(ACTIVATIONS, start=11)
]
plot_files += [
    f"{index:02d}_overall_{metric.lower()}.png"
    for index, metric in enumerate(metric_names, start=14)
]
plot_files += ["18_computational_resource_comparison.png"]

In [22]:
results_df.sort_values(
    "Validation F1-Score", ascending=False
).to_csv(
    os.path.join(RESULTS_DIR, "ann_ranked_72_experiments.csv"),
    index=False
)

resource_columns = [
    "Model", "Hidden Layers", "Activation", "Learning Rate",
    "Training Time (s)", "Prediction Time (s)", "CPU Time (s)",
    "CPU Utilization Snapshot (%)", "Memory Before (MB)",
    "Memory After (MB)", "Memory Change (MB)"
]
results_df[resource_columns].to_csv(
    os.path.join(RESULTS_DIR, "ann_resource_comparison.csv"),
    index=False
)


In [23]:
validation_report = classification_report(
    y_val, best_validation_prediction,
    labels=class_labels, zero_division=0
)
with open(
    os.path.join(RESULTS_DIR, "best_ann_validation_report.txt"),
    "w", encoding="utf-8"
) as f:
    f.write(f"Selected configuration: {best_name}\n\n")
    f.write(validation_report)

with open(
    os.path.join(RESULTS_DIR, "experiment_summary.txt"),
    "w", encoding="utf-8"
) as f:
    f.write("ANN WINE QUALITY CLASSIFICATION\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Dataset after duplicate removal: {df.shape}\n")
    f.write(f"Target classes: {class_labels}\n")
    f.write("Data split: 60% training, 20% validation, 20% test (approx.)\n")
    f.write(f"Hidden layers: {HIDDEN_LAYER_COUNTS}\n")
    f.write(f"Neurons per hidden layer: {NEURONS_PER_LAYER}\n")
    f.write(f"Activation functions: {ACTIVATIONS}\n")
    f.write(f"Learning rates: {LEARNING_RATES}\n")
    f.write(f"Total ANN configurations: {len(results_df)}\n")
    f.write(f"Maximum iterations: {MAX_ITER}\n")
    f.write(f"Total experiment time: {total_execution_time:.3f} seconds\n\n")
    f.write(f"Best configuration selected on validation F1: {best_name}\n\n")
    f.write("Held-out test metrics:\n")
    for metric, value in test_metrics.items():
        f.write(f"{metric}: {value:.6f}\n")
    f.write(f"Test prediction time: {best_test_prediction_time:.6f} seconds\n\n")
    f.write("Classification metrics: Accuracy, weighted Precision, ")
    f.write("weighted Recall, weighted F1-score, Confusion Matrix.\n")
    f.write("Resource metrics: training time, prediction time, CPU time, ")
    f.write("CPU utilization snapshot, memory before/after/change.\n")
    f.write("\nThe 72 configurations are compared using validation metrics. ")
    f.write("The selected configuration is refitted on training + validation ")
    f.write("data and evaluated once on the held-out test set.\n")

In [24]:
missing_plots = [
    name for name in plot_files
    if not os.path.exists(os.path.join(RESULTS_DIR, name))
]

print("\n" + "=" * 75)
print("ANN CLASSIFICATION COMPLETED")
print("=" * 75)
print("\nSelected ANN configuration:", best_name)
print("\nHeld-out test metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print(f"\nTotal configurations evaluated: {len(results_df)}")
print(f"Total experiment execution time: {total_execution_time:.2f} seconds")
print(f"Expected plot images: {len(plot_files)}")
print("Missing plot files:", missing_plots)
print("Results folder:", RESULTS_DIR)

print("\nGenerated plots:")
for name in plot_files:
    print("-", name)

print("\nAll 72 experiment results:")
print(os.path.join(RESULTS_DIR, "ann_all_72_experiments.csv"))



ANN CLASSIFICATION COMPLETED

Selected ANN configuration: ANN_L2_relu_LR0.0005

Held-out test metrics:
Accuracy: 0.5338
Precision: 0.5080
Recall: 0.5338
F1-Score: 0.5178

Total configurations evaluated: 72
Total experiment execution time: 736.58 seconds
Expected plot images: 18
Missing plot files: []
Results folder: ANN_classification_results

Generated plots:
- 01_tsne_visualization.png
- 02_best_ann_confusion_matrix.png
- 03_hidden_layers_1.png
- 04_hidden_layers_2.png
- 05_hidden_layers_3.png
- 06_hidden_layers_4.png
- 07_hidden_layers_5.png
- 08_hidden_layers_6.png
- 09_hidden_layers_7.png
- 10_hidden_layers_8.png
- 11_activation_relu.png
- 12_activation_tanh.png
- 13_activation_logistic.png
- 14_overall_accuracy.png
- 15_overall_precision.png
- 16_overall_recall.png
- 17_overall_f1-score.png
- 18_computational_resource_comparison.png

All 72 experiment results:
ANN_classification_results\ann_all_72_experiments.csv
